In [1]:
import sys
import os

CURRENT_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
from agents import User, Module, Orchestrator
from utils import InstructRecDataset, ItemIndex
from recsys.llms import create_recsys, user_message
from advisor import create_advisor

from capabilities.memories import SharedMemory, MemoryContent
from capabilities.skills import SharedMemorySkill

In [3]:
ds = InstructRecDataset(PROJECT_ROOT + "/datasets/instructrec/booksAll_recagent.pkl")
item_index = ItemIndex(PROJECT_ROOT + "/datasets/instructrec/combined_books_asin_mapping.csv")

In [4]:
user_id = 0 
item = item_index.get_item_by_index(5915) # Example item index

print(item["title"])
print(item["description"])
print(item["asin"])

Like a Watered Garden: A Novel (Garden Gates)
['From the opening line ("I received a box of flowers from my dead husband"), debut novelist Hill hooks the reader with this intriguing look at love, faith, grieving and relationships. The unusually named Mibby Garrett is a garden designer who runs Perennially Yours, serving her clients\' eclectic landscaping needs from behind the wheel of the Daisy Mobile with her companionable dog, Blink. The redhead\'s life is turned topsy-turvy when her husband is unexpectedly killed; she can barely cope with her business and with her 13-year-old son, Kyle. Her next-door neighbor, well-heeled 30-something Louise Giovanelli, keeps Mibby sane, supplying her with love, advice and baked goods from the B&amp;B she owns next door. Odd guests at the B&amp;B and Mibby\'s eccentric garden customers, including a handsome widower, are competently portrayed. When Mibby discovers her husband had a secret past, it threatens to destroy her last shreds of faith. Althou

In [5]:
MAX_HISTORY = 3
reviews = ds.get_review_text(user_id)
titles = ds.get_titles(user_id)
descriptions = ds.get_descriptions(user_id)
instruction = ds.get_instruction(user_id)
persona = ds.get_persona(user_id)
asins = ds.get_asins(user_id) # Histórico do usuario
ranked_lists = ds.get_ranked_list(user_id) #Lista ranqueada de recomendação. Nao usar. nao faz sentido para agora

user_payload = {
    "user_id": user_id,
    "persona": persona,
    "instruction": instruction,
    "history": [
        {
            "asin": asin,
            "title": title,
            "description": desc,
            "review": review,
        }
        for asin, title, desc, review in list(zip(asins, titles, descriptions, reviews))[:MAX_HISTORY]
    ],
}

In [6]:
read_books = ""
for id, title, desc, review in list(zip(asins, titles, descriptions, reviews))[:MAX_HISTORY]:
    read_books += (
        f"- ID: {id}\n"
        f"  Name: {title}\n"
        f"  Description: {desc}\n"
        f"  User Review: {review}\n\n"

    )


user_message = user_message.format(
    persona=persona,
    read_books=read_books,
    instruction=instruction
)

shared_memory = SharedMemorySkill(shared_memory=SharedMemory(name="shared_chat_history"))
user = User(name ="user", skills=[shared_memory])
recsys_orchestrator = create_recsys()
advisor = create_advisor(shared_memory, recsys=recsys_orchestrator)

id_a = user.add_shared_memory(
    MemoryContent(content=user_payload),
)

communication_mode = "diagnostic"  # ou "advisory" ou "mediated"

# 2. Module com transições baseadas no modo
if communication_mode == "diagnostic":
    transitions = {
        user: [recsys_orchestrator],           # user só fala com recsys
        recsys_orchestrator: [advisor],         # recsys manda pro advisor
        advisor: [user],                        # advisor responde ao user
    }
elif communication_mode == "advisory":
    transitions = {
        user: [advisor],
        advisor: [user],
    }
elif communication_mode == "mediated":
    transitions = {
        user: [advisor],
        advisor: [recsys_orchestrator, user],   # advisor pode ir pro recsys ou user
        recsys_orchestrator: [advisor],
    }

# 3. Module principal
main_module = Module(
    name="main_module",
    agents=[user, recsys_orchestrator, advisor],
    speaker_selection_method="auto",
    allowed_or_disallowed_speaker_transitions=transitions,
    speaker_transitions_type="allowed",
)

# 4. Orchestrator principal
main_orchestrator = Orchestrator(
    name="main_orchestrator",
    module=main_module
)


In [7]:
user.talk_to(main_orchestrator, message = user_message)
'''
user.talk_to(advisor, message = """
{
  "chatgpt": {
    "recommendation": {
      "item_id": 88888,
      "title": "A New Comforting Story",
      "reason": "Narrativa leve e previsível, alinhada ao desejo de escapismo do usuário."
    },
    "explanation_items_ordered": [107208, 556],
    "baseline_ranking": [88888, 77777, 66666, 55555]
  },
  "gemini-2.5-flash": {
    "recommendation": {
      "item_id": 99999,
      "title": "Warm Roads Ahead",
      "reason": "Livro feel-good com ritmo estável, compatível com as preferências históricas."
    },
    "explanation_items_ordered": [556, 57570],
    "baseline_ranking": [99999, 44444, 33333, 22222]
  }
}
""")
'''


user ⟶ main_orchestrator:

USER PERSONA
A filmmaker working on a project about the impact of cars on society

---

BOOKS PREVIOUSLY READ BY THE USER
Each entry contains the book id, book title, a factual description, and the user's own review.
When referencing past items, always use the ID field.

- ID: 107208
  Name: The Executive's Decision: The Keller Family Series
  Description: ['Bernadette Marie grew up obsessed with pens and notebooks, each one filled with lists and ideas for stories. Not much has changed. This wife and mother of five sons has a passion for writing stories about falling in love, finding love where you left it, and strong families. Bernadette Marie is an accomplished martial artist who holds a Black Belt in Tang Soo Do and she is a chronic entrepreneur. She is a member of Romance Writers of America and Colorado Romance Writers. Visit her website at www.bernadettemarie.com for news on upcoming releases, signings, appearances, and contests.', '', '']
  User Review:

/Users/fillipesantos/Documents/projects/arara/src/capabilities/clients/utils/calculate_token_cost.py:111: UserWarning: Exact match for model 'google/gemini-2.5-flash-lite' not found for provider 'openrouter'. Using pricing for base model 'google/gemini-2.5-flash'.
  warnings.warn(


claude_3_5_sonnet ⟶ aggregator:
{
  "top_k_recommendations": [
    {
      "rank": 1,
      "book_title": "Running in Heels",
      "explanation": "Based on your interest in everyday narratives and athletic themes, this light-hearted story combines fashion and personal growth, offering the kind of escapist entertainment you seek as a filmmaker looking for relief from serious societal topics.",
      "used_history_asins": ["107208"]
    },
    {
      "rank": 2,
      "book_title": "The Yoga Studio",
      "explanation": "Given your appreciation for stories about everyday people as shown in your review of 'Gumbeaux', this feel-good narrative about a yoga community provides the predictable yet engaging escape you're seeking.",
      "used_history_asins": ["556"]
    },
    {
      "rank": 3,
      "book_title": "Dance of Life",
      "explanation": "Drawing from your interest in unique narratives as shown in your review of 'The Hummingbird Wizard', this story offers an entertaining blend

'\nuser.talk_to(advisor, message = """\n{\n  "chatgpt": {\n    "recommendation": {\n      "item_id": 88888,\n      "title": "A New Comforting Story",\n      "reason": "Narrativa leve e previsível, alinhada ao desejo de escapismo do usuário."\n    },\n    "explanation_items_ordered": [107208, 556],\n    "baseline_ranking": [88888, 77777, 66666, 55555]\n  },\n  "gemini-2.5-flash": {\n    "recommendation": {\n      "item_id": 99999,\n      "title": "Warm Roads Ahead",\n      "reason": "Livro feel-good com ritmo estável, compatível com as preferências históricas."\n    },\n    "explanation_items_ordered": [556, 57570],\n    "baseline_ranking": [99999, 44444, 33333, 22222]\n  }\n}\n""")\n'

In [8]:
user._oai_messages

defaultdict(list,
            {<agents.orchestrator.Orchestrator at 0x124f74260>: [{'content': "\nUSER PERSONA\nA filmmaker working on a project about the impact of cars on society\n\n---\n\nBOOKS PREVIOUSLY READ BY THE USER\nEach entry contains the book id, book title, a factual description, and the user's own review.\nWhen referencing past items, always use the ID field.\n\n- ID: 107208\n  Name: The Executive's Decision: The Keller Family Series\n  Description: ['Bernadette Marie grew up obsessed with pens and notebooks, each one filled with lists and ideas for stories. Not much has changed. This wife and mother of five sons has a passion for writing stories about falling in love, finding love where you left it, and strong families. Bernadette Marie is an accomplished martial artist who holds a Black Belt in Tang Soo Do and she is a chronic entrepreneur. She is a member of Romance Writers of America and Colorado Romance Writers. Visit her website at www.bernadettemarie.com for news o